In [ ]:
# Setup — all imports live here so the notebook executes top-to-bottom
# without reimporting mid-notebook. When running headlessly (CI, HPC),
# uncomment the ``matplotlib.use('Agg')`` line *before* the pyplot import
# so figures never need an interactive display.
import dataclasses
import logging
import os
import sys
from pathlib import Path

# import matplotlib
# matplotlib.use('Agg')  # enable for headless runs
import matplotlib.pyplot as plt
import mne
import numpy as np
import seaborn as sns
from mne.viz import plot_topomap

# Resolve project root regardless of where the notebook is launched from
for _candidate in [".", "..", "../.."]:  # noqa: B007
    _p = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        break

from sklearn.decomposition import PCA, FastICA  # noqa: E402

from scripts.analysis_common import (  # noqa: E402
    FREQUENCY_BANDS,
    WAVELET_BAND_FREQ_RESOLUTION_HZ,
    analyzers_to_datasets,
    load_analyzers,
    wavelet_transform,
)
from src.analysis.wavelet_ica import zscore_by_time  # noqa: E402
from src.definitions.constants import ProjectPaths  # noqa: E402
from src.definitions.fields import (  # noqa: E402
    ConditionVariants,
    ExclusionCategories,
    MusicTypeVariants,
)

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.0)
%matplotlib inline

# Subject-Frequency ICA / PCA on Wavelet Power

## Approach overview

**Approach 5 — "Subject-Frequency" decomposition.**  We concatenate
*subject × frequency* into the observation (sample) axis and keep
*channel × time* as the feature axis:

```
Input:   (n_subjects, n_channels, n_freqs, n_times)  — 4-D wavelet power
Reshape: (n_subjects × n_freqs,  n_channels × n_times)
         ──── observations ────  ──── features ───────
```

The resulting 2-D matrix has **S × F rows** (one per subject–frequency pair)
and **C × T columns** (one per channel–time-point pair).  Each row is a
full spatial-temporal power surface at a single frequency for a single subject.

### What the decomposition finds

PCA / ICA applied to this matrix discover a small set of **spatial-temporal
components** — recurring channel × time patterns that appear across
subjects and frequencies.

| Quantity | Shape | Interpretation |
|----------|-------|----------------|
| **Component pattern** | `(n_channels, n_times)` | A spatial-temporal template: which electrodes and time points co-activate in this component |
| **Component topography** | `(n_channels,)` | Channel marginal — mean absolute loading over time → scalp map |
| **Time course** | `(n_times,)` | Time marginal — mean absolute loading over channels → temporal profile |
| **Activation scores** | `(n_subjects, n_freqs)` | How strongly each component is expressed at each frequency for each subject |

### Interpretation guide

- A component whose channel × time map shows strong loading in
  **posterior channels during early time windows** indicates a stereotypical
  sensory processing topography.
- The per-subject frequency profiles show **which frequencies** drive each
  component for each participant.  If the profiles are similar across
  subjects, the component captures a shared spectral signature.
- Components with **high inter-individual similarity** in frequency profiles
  are the most interesting: they represent brain patterns consistently
  triggered across participants at specific frequency bands.

### Analyses

1. PCA scree plot (variance explained)
2. PCA channel × time component maps
3. Per-subject frequency profiles
4. PCA topographic maps (time-averaged channel marginal)
5. Cross-component correlation (PCA vs ICA)
6. ICA channel × time component maps
7. ICA topomaps — mean channel loading across time
8. ICA topomaps — variance of channel loadings across time
9. ICA time courses (channel-averaged)
10. Inter-individual IC correlations (frequency-based)

## Configuration

In [ ]:
# ── Experiment configuration ─────────────────────────────────────────────────
CONDITION = ConditionVariants.PLACEBO
MUSIC_TYPES = [MusicTypeVariants.CLASSICAL]  # single type for fast exploration
EXCLUSION_CATEGORIES = [ExclusionCategories.BAD_MUSIC, ExclusionCategories.ARTIFACTS]
PROCESS_AND_SAVE_DATA = False

# ── Wavelet settings ─────────────────────────────────────────────────────────
REPRESENTATION = "power"
WAVELET_FREQ_MIN = min(lo for lo, _ in FREQUENCY_BANDS.values())
WAVELET_FREQ_MAX = max(hi for _, hi in FREQUENCY_BANDS.values())
WAVELET_N_FREQS = max(
    2,
    int(round((WAVELET_FREQ_MAX - WAVELET_FREQ_MIN) / WAVELET_BAND_FREQ_RESOLUTION_HZ))
    + 1,
)
FREQS = np.linspace(WAVELET_FREQ_MIN, WAVELET_FREQ_MAX, WAVELET_N_FREQS)

KEEP_FREQUENCY_DIM = True
RESHAPE_FREQUENCY_DIM = True

# ── Reuse / compute ──────────────────────────────────────────────────────────
REUSE_WAVELETS = True

# ── Subsets ────────────────────────────────────────────────────────────────────
N_SUBJECTS_SUBSET: int | None = 5
N_CHANNELS_SUBSET: int | None = 32
N_TIMES_SUBSET: int | None = 10000

# ── Decomposition settings ────────────────────────────────────────────────────
N_COMPONENTS_PCA = 20  # number of PCA components to retain
N_COMPONENTS_ICA = 10  # number of ICA components to extract
ICA_RANDOM_STATE = 42

# ── Storage directory ─────────────────────────────────────────────────────────
WAVELET_DIR: Path = (
    ProjectPaths.NOTEBOOKS_DIR / "04-wavelet-ica-analysis" / "wavelet_cache"
)

# ── Plot saving ──────────────────────────────────────────────────────────────
SAVE_PLOTS = True
# Canonical notebook plot output layout (mirrors the production CLI
# script scripts/run_wavelet_ica.py):
#   notebooks/04-wavelet-ica-analysis/plots/subject_frequency/pca_ica/
PLOTS_DIR = (
    ProjectPaths.NOTEBOOKS_DIR
    / "04-wavelet-ica-analysis"
    / "plots"
    / "subject_frequency"
    / "pca_ica"
)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Wavelet base directory : {WAVELET_DIR}")
print(
    f"Frequencies            : {FREQS[0]:.1f}–{FREQS[-1]:.1f} Hz ({len(FREQS)} steps)"
)
print(f"PCA components         : {N_COMPONENTS_PCA}")
print(f"ICA components         : {N_COMPONENTS_ICA}")

## Data Loading

In [ ]:
analyzers = load_analyzers(
    MUSIC_TYPES,
    CONDITION,
    EXCLUSION_CATEGORIES,
    PROCESS_AND_SAVE_DATA,
    normalize_data=False,
)
datasets = analyzers_to_datasets(analyzers)

if N_SUBJECTS_SUBSET is not None:
    datasets = {
        label: dataclasses.replace(ad, data=ad.data[:N_SUBJECTS_SUBSET])
        for label, ad in datasets.items()
    }
    print(f"Using first {N_SUBJECTS_SUBSET} individuals.")

if N_CHANNELS_SUBSET is not None:
    datasets = {
        label: dataclasses.replace(ad, data=ad.data[:, :N_CHANNELS_SUBSET, :])
        for label, ad in datasets.items()
    }
    print(f"Using first {N_CHANNELS_SUBSET} channels.")

if N_TIMES_SUBSET is not None:
    datasets = {
        label: dataclasses.replace(ad, data=ad.data[:, :, :N_TIMES_SUBSET])
        for label, ad in datasets.items()
    }
    print(f"Using first {N_TIMES_SUBSET} time samples.")

print("Loaded datasets:", list(datasets.keys()))
for label, ad in datasets.items():
    print(
        f"  {label}: {ad.n_items} subjects, {ad.n_features} channels, "
        f"{ad.n_samples} samples"
    )

## Load or Compute Wavelet Transforms

Stored in `WAVELET_DIR/broadband/`.

In [ ]:
broadband_datasets = wavelet_transform(
    datasets=datasets,
    freqs=FREQS,
    representation=REPRESENTATION,
    keep_frequency_dim=KEEP_FREQUENCY_DIM,
    reshape_frequency_dim=RESHAPE_FREQUENCY_DIM,
    wavelet_dir=WAVELET_DIR / "broadband",
    reuse_wavelets=REUSE_WAVELETS,
)
for label, ad in broadband_datasets.items():
    source = ad.metadata.get("loaded_from_wavelet_file", "computed_now")
    print(f"[broadband] {label}: shape={ad.data.shape}  source={source}")

## Dataset Selection

Change `LABEL` to switch between music types.

In [ ]:
LABEL = list(broadband_datasets.keys())[0]

bb_ad = broadband_datasets[LABEL]
bb_data = bb_ad.data  # (n_subjects, n_channels, n_freqs, n_times)
sfreq = bb_ad.sfreq

n_subjects, n_channels, n_freqs, n_times = bb_data.shape
time = np.arange(n_times) / sfreq

print(f"Dataset    : {LABEL}")
print(
    f"Shape      : {bb_data.shape}  (subjects × channels × freqs × times)"
)
print(f"Duration   : {n_times / sfreq:.1f} s  @  {sfreq} Hz")
print(f"Freq range : {FREQS[0]:.1f}–{FREQS[-1]:.1f} Hz ({n_freqs} steps)")

## Subject-Frequency Reshape

Transpose to `(S, F, C, T)` then flatten into a 2-D matrix with
`S × F` observation rows and `C × T` feature columns.

```
X_sf:  (n_subjects × n_freqs,  n_channels × n_times)
```

Each row is one (subject, frequency) observation of the full
channel × time power surface.  PCA/ICA will discover recurring
spatial-temporal patterns shared across subjects and frequencies.

Before reshaping we **z-score along time** (axis=-1) so that each
`(subject, channel, frequency)` slice has zero mean and unit variance.
This matches the production pipeline in `src/analysis/wavelet_ica.py`.

In [ ]:
# Z-score along time: each (subject, channel, frequency) slice → mean=0, std=1
bb_z = zscore_by_time(bb_data)  # (S, C, F, T)

# Transpose to (S, F, C, T) then flatten to (S*F, C*T)
bb_transposed = bb_z.transpose(0, 2, 1, 3)  # (S, F, C, T)
X_sf = bb_transposed.reshape(n_subjects * n_freqs, n_channels * n_times)
print(f"Subject-Frequency matrix shape: {X_sf.shape}")
print(f"  Rows = {n_subjects} subjects × {n_freqs} frequencies")
print(f"  Cols = {n_channels} channels × {n_times} time points")

---
## 1 — PCA Scree Plot (Variance Explained)

The intrinsic dimensionality of the channel × time space tells us how
many spatial-temporal patterns are needed to summarise the data.

In [ ]:
pca = PCA(n_components=N_COMPONENTS_PCA, random_state=ICA_RANDOM_STATE)
pca.fit(X_sf)  # (n_obs, n_features)

explained = pca.explained_variance_ratio_
cumulative = np.cumsum(explained)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(1, len(explained) + 1), explained, color="steelblue")
axes[0].set_xlabel("Component")
axes[0].set_ylabel("Explained variance ratio")
axes[0].set_title("Individual")

axes[1].plot(range(1, len(cumulative) + 1), cumulative, "o-", color="darkorange")
axes[1].axhline(0.90, ls="--", color="grey", alpha=0.6)
axes[1].set_xlabel("Component")
axes[1].set_ylabel("Cumulative variance")
axes[1].set_title("Cumulative")

fig.suptitle(f"PCA Scree — Subject-Frequency — {LABEL}", fontsize=13)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / f"pca_scree_{LABEL}.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")

print(f"Total variance in {N_COMPONENTS_PCA} components: {cumulative[-1]:.3f}")

---
## 2 — PCA Channel × Time Component Maps

Each PCA component is a vector of length `n_channels × n_times`.
Reshape it to `(n_channels, n_times)` and display as a heatmap.
Columns encode the spatial-temporal pattern of the component.

In [ ]:
# PCA components: (K, n_channels * n_times) → (K, C, T)
components = pca.components_  # (K, C*T)

n_show = min(6, N_COMPONENTS_PCA)
components_2d = components[:n_show].reshape(n_show, n_channels, n_times)
time_extent = n_times / sfreq

n_cols = min(3, n_show)
n_rows = int(np.ceil(n_show / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
flat = np.array(axes).flatten() if n_show > 1 else [axes]
for i in range(n_show):
    ax = flat[i]
    im = ax.imshow(
        components_2d[i],
        aspect="auto",
        origin="lower",
        extent=[0, time_extent, 0, n_channels],
        cmap="RdBu_r",
    )
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Channel index")
    ax.set_title(f"PC {i + 1}")
    fig.colorbar(im, ax=ax)
for i in range(n_show, len(flat)):
    flat[i].set_visible(False)
fig.suptitle(f"PCA Channel × Time Maps — Subject-Frequency — {LABEL}", fontsize=13)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(
        PLOTS_DIR / f"pca_channel_time_maps_{LABEL}.png",
        dpi=150,
        bbox_inches="tight",
    )
plt.show()
plt.close("all")

---
## 3 — Per-Subject Frequency Profiles

Reshape PCA scores `(S×F, K)` to `(S, F, K)`.  For each component,
plot the frequency profile per subject — this shows which frequencies
drive each component for each participant.

In [ ]:
# PCA scores: (S*F, K) → (S, F, K)
scores = pca.transform(X_sf)
scores_3d = scores.reshape(n_subjects, n_freqs, N_COMPONENTS_PCA)

n_show = min(4, N_COMPONENTS_PCA)
fig, axes = plt.subplots(n_show, 1, figsize=(10, 2.5 * n_show), sharex=True)
if n_show == 1:
    axes = [axes]
for i, ax in enumerate(axes):
    for s in range(n_subjects):
        ax.plot(FREQS, scores_3d[s, :, i], lw=0.8, alpha=0.6, label=f"S{s + 1}")
    ax.set_ylabel(f"PC {i + 1}")
axes[-1].set_xlabel("Frequency (Hz)")
axes[0].legend(fontsize=6, ncol=min(n_subjects, 8), loc="upper right")
fig.suptitle(f"Per-Subject Frequency Profiles — {LABEL}", fontsize=13)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(
        PLOTS_DIR / f"pca_subject_freq_profiles_{LABEL}.png",
        dpi=150,
        bbox_inches="tight",
    )
plt.show()
plt.close("all")

---
## 4 — PCA Topographic Maps (Time-Averaged Channel Marginal)

Marginalize the component vectors over time to get a per-channel loading:
`channel_marginal[k, c] = mean(|component[k, c, :]|)`.  Display as a
scalp topomap.

In [ ]:
# Channel marginal: mean loading over time → (K, C)
components_full = components.reshape(N_COMPONENTS_PCA, n_channels, n_times)
channel_marginal = components_full.mean(axis=2)  # (K, C)

n_show = min(6, N_COMPONENTS_PCA)

info = bb_ad.mne_info
if info is not None and len(info.ch_names) >= n_channels:
    _info = mne.pick_info(info, sel=range(n_channels))
    n_cols = min(3, n_show)
    n_rows = int(np.ceil(n_show / n_cols))
    fig, axes = plt.subplots(
        n_rows, n_cols, figsize=(4 * n_cols, 3.5 * n_rows)
    )
    flat = np.array(axes).flatten() if n_show > 1 else [axes]
    for i in range(n_show):
        plot_topomap(channel_marginal[i], _info, axes=flat[i], show=False)
        flat[i].set_title(f"PC {i + 1}")
    for i in range(n_show, len(flat)):
        flat[i].set_visible(False)
    fig.suptitle(
        f"PCA Topomaps (time-averaged) — Subject-Frequency — {LABEL}", fontsize=13
    )
    fig.tight_layout()
    if SAVE_PLOTS:
        fig.savefig(
            PLOTS_DIR / f"pca_topomaps_{LABEL}.png", dpi=150, bbox_inches="tight"
        )
    plt.show()
    plt.close("all")
else:
    print("No MNE Info available — skipping topomaps.")

---
## 5 — Cross-Component Correlation (PCA vs ICA)

Fit FastICA on the same data and compare the correlation structure
between PCA scores and ICA sources.

In [ ]:
ica = FastICA(
    n_components=N_COMPONENTS_ICA,
    random_state=ICA_RANDOM_STATE,
    max_iter=1000,
    whiten="unit-variance",
)
ica_sources = ica.fit_transform(X_sf)  # (S*F, K_ica)
ica_mixing = ica.mixing_  # (C*T, K_ica)

# Correlation: PCA scores vs ICA sources
pca_ica = np.corrcoef(scores.T, ica_sources.T)
n_pca = scores.shape[1]
n_ica = ica_sources.shape[1]
cross = pca_ica[:n_pca, n_pca:]  # (K_pca, K_ica)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(np.abs(cross), vmin=0, vmax=1, cmap="viridis", aspect="auto")
ax.set_xlabel("ICA component")
ax.set_ylabel("PCA component")
ax.set_xticks(range(n_ica))
ax.set_xticklabels([f"IC{i + 1}" for i in range(n_ica)], fontsize=7)
ax.set_yticks(range(n_pca))
ax.set_yticklabels([f"PC{i + 1}" for i in range(n_pca)], fontsize=7)
fig.colorbar(im, ax=ax, label="|correlation|")
fig.suptitle(f"PCA vs ICA — Subject-Frequency — {LABEL}", fontsize=13)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(
        PLOTS_DIR / f"cross_component_correlation_{LABEL}.png",
        dpi=150,
        bbox_inches="tight",
    )
plt.show()
plt.close("all")

---
## 6 — ICA Channel × Time Component Maps

Reshape ICA mixing matrix columns `(C×T, K_ica)` into `(C, T, K_ica)`
and display as channel × time heatmaps.

In [ ]:
# ICA mixing: (C*T, K_ica) → (C, T, K_ica)
ica_mixing_ct = ica_mixing.reshape(n_channels, n_times, N_COMPONENTS_ICA)

n_show = min(6, N_COMPONENTS_ICA)
n_cols = min(3, n_show)
n_rows = int(np.ceil(n_show / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
flat = np.array(axes).flatten() if n_show > 1 else [axes]
for i in range(n_show):
    ax = flat[i]
    im = ax.imshow(
        ica_mixing_ct[:, :, i],
        aspect="auto",
        origin="lower",
        extent=[0, time_extent, 0, n_channels],
        cmap="RdBu_r",
    )
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Channel index")
    ax.set_title(f"IC {i + 1}")
    fig.colorbar(im, ax=ax)
for i in range(n_show, len(flat)):
    flat[i].set_visible(False)
fig.suptitle(f"ICA Channel × Time Maps — {LABEL}", fontsize=13)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(
        PLOTS_DIR / f"ica_channel_time_maps_{LABEL}.png",
        dpi=150,
        bbox_inches="tight",
    )
plt.show()
plt.close("all")

---
## 7 — ICA Topographic Maps — Mean Loading Across Time

Average the ICA mixing over time for each channel to get
the mean spatial pattern of each IC.

In [ ]:
# Time-averaged ICA channel loadings: mean over T → (C, K_ica)
ica_ch_mean = ica_mixing_ct.mean(axis=1)  # (C, K_ica)

n_show = min(6, N_COMPONENTS_ICA)

if info is not None and len(info.ch_names) >= n_channels:
    _info = mne.pick_info(info, sel=range(n_channels))
    n_cols = min(3, n_show)
    n_rows = int(np.ceil(n_show / n_cols))
    fig, axes = plt.subplots(
        n_rows, n_cols, figsize=(4 * n_cols, 3.5 * n_rows)
    )
    flat = np.array(axes).flatten() if n_show > 1 else [axes]
    for i in range(n_show):
        plot_topomap(ica_ch_mean[:, i], _info, axes=flat[i], show=False)
        flat[i].set_title(f"IC {i + 1} (mean)")
    for i in range(n_show, len(flat)):
        flat[i].set_visible(False)
    fig.suptitle(
        f"ICA Topomaps — Mean Loading — {LABEL}", fontsize=13
    )
    fig.tight_layout()
    if SAVE_PLOTS:
        fig.savefig(
            PLOTS_DIR / f"ica_topomap_mean_{LABEL}.png",
            dpi=150,
            bbox_inches="tight",
        )
    plt.show()
    plt.close("all")
else:
    print("No MNE Info available — skipping topomaps.")

---
## 8 — ICA Topographic Maps — Variance Across Time

Compute the variance of the ICA mixing over time for each channel.
High variance indicates channels whose contribution to the IC
fluctuates most over the recording.

In [ ]:
# Variance of ICA channel loadings across time → (C, K_ica)
ica_ch_var = ica_mixing_ct.var(axis=1)  # (C, K_ica)

n_show = min(6, N_COMPONENTS_ICA)

if info is not None and len(info.ch_names) >= n_channels:
    _info = mne.pick_info(info, sel=range(n_channels))
    n_cols = min(3, n_show)
    n_rows = int(np.ceil(n_show / n_cols))
    fig, axes = plt.subplots(
        n_rows, n_cols, figsize=(4 * n_cols, 3.5 * n_rows)
    )
    flat = np.array(axes).flatten() if n_show > 1 else [axes]
    for i in range(n_show):
        plot_topomap(ica_ch_var[:, i], _info, axes=flat[i], show=False)
        flat[i].set_title(f"IC {i + 1} (var)")
    for i in range(n_show, len(flat)):
        flat[i].set_visible(False)
    fig.suptitle(
        f"ICA Topomaps — Variance — {LABEL}", fontsize=13
    )
    fig.tight_layout()
    if SAVE_PLOTS:
        fig.savefig(
            PLOTS_DIR / f"ica_topomap_variance_{LABEL}.png",
            dpi=150,
            bbox_inches="tight",
        )
    plt.show()
    plt.close("all")
else:
    print("No MNE Info available — skipping topomaps.")

---
## 9 — ICA Time Courses (Channel-Averaged)

Average the ICA mixing matrix `(C, T, K_ica)` over channels to get a
single time course for each IC.  This shows the temporal dynamics of each
independent component.

In [ ]:
# Channel-averaged ICA time profiles: mean over C → (T, K_ica) → transpose
ica_time_profiles = ica_mixing_ct.mean(axis=0).T  # (K_ica, T)

n_ica = ica_time_profiles.shape[0]
time = np.arange(n_times) / sfreq

fig, axes = plt.subplots(n_ica, 1, figsize=(14, 2 * n_ica), sharex=True)
if n_ica == 1:
    axes = [axes]
for i, ax in enumerate(axes):
    ax.plot(time, ica_time_profiles[i], lw=0.6, color="steelblue")
    ax.set_ylabel(f"IC {i + 1}")
axes[-1].set_xlabel("Time (s)")
fig.suptitle(f"ICA Time Courses (channel-averaged) — {LABEL}", fontsize=13)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(
        PLOTS_DIR / f"ica_timecourses_{LABEL}.png", dpi=150, bbox_inches="tight"
    )
plt.show()
plt.close("all")

---
## 10 — Inter-Individual IC Correlations

Reshape ICA sources `(S×F, K_ica)` to `(S, F, K_ica)`.  For each IC,
compute the Pearson correlation between every pair of subjects over
frequencies.  High correlations indicate that two subjects have similar
frequency profiles for that IC — i.e. the component is consistently
expressed across individuals.

In [ ]:
# ICA sources: (S*F, K_ica) → (S, F, K_ica)
ica_scores_3d = ica_sources.reshape(n_subjects, n_freqs, N_COMPONENTS_ICA)

n_show = min(6, N_COMPONENTS_ICA)
fig, axes = plt.subplots(
    1, n_show, figsize=(3.5 * n_show, 3.5), constrained_layout=True
)
if n_show == 1:
    axes = [axes]
for k in range(n_show):
    comp_data = ica_scores_3d[:, :, k]  # (S, F)
    corr_matrix = np.corrcoef(comp_data)  # (S, S)
    im = axes[k].imshow(corr_matrix, vmin=-1, vmax=1, cmap="RdBu_r")
    axes[k].set_title(f"IC {k + 1}")
    axes[k].set_xlabel("Subject")
    axes[k].set_ylabel("Subject")
fig.colorbar(im, ax=axes[-1], fraction=0.046)
fig.suptitle(f"Inter-Individual IC Correlations — {LABEL}", fontsize=13)
if SAVE_PLOTS:
    fig.savefig(
        PLOTS_DIR / f"ica_interindividual_correlation_{LABEL}.png",
        dpi=150,
        bbox_inches="tight",
    )
plt.show()
plt.close("all")

---
## Summary

Available variables for further analysis:

| Variable | Shape | Description |
|----------|-------|-------------|
| `X_sf` | `(S×F, C×T)` | Z-scored subject-frequency matrix |
| `pca` | — | Fitted PCA object |
| `scores` | `(S×F, K)` | PCA component scores |
| `scores_3d` | `(S, F, K)` | PCA scores reshaped per-subject |
| `components` | `(K, C×T)` | PCA loading vectors |
| `channel_marginal` | `(K, C)` | Time-averaged PCA channel loadings |
| `ica` | — | Fitted FastICA object |
| `ica_sources` | `(S×F, K_ica)` | ICA source activations |
| `ica_scores_3d` | `(S, F, K_ica)` | ICA scores reshaped per-subject |
| `ica_mixing` | `(C×T, K_ica)` | ICA mixing matrix |
| `ica_mixing_ct` | `(C, T, K_ica)` | ICA mixing reshaped to channel × time |
| `ica_ch_mean` | `(C, K_ica)` | Time-averaged ICA channel loadings |
| `ica_ch_var` | `(C, K_ica)` | Variance of ICA channel loadings (time) |
| `ica_time_profiles` | `(K_ica, T)` | Channel-averaged ICA time courses |